# Analyse exploratoire : circulations, annulations et retards TGV

L'objectif de ce notebook est d'explorer le jeu de données mensuel de régularité TGV (SNCF Open Data), de repérer les anomalies de qualité à traiter avant modélisation, et d'examiner la relation entre le volume de trains prévus sur une liaison et ses taux d'annulation et de retard à l'arrivée.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

CHEMIN_CSV = Path("../data/raw/regularite-mensuelle-tgv-aqst.csv")
if not CHEMIN_CSV.exists():
    CHEMIN_CSV = Path("data/raw/regularite-mensuelle-tgv-aqst.csv")

df = pd.read_csv(CHEMIN_CSV, sep=";")
print(f"Dimensions : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"Période couverte : {df['Date'].min()} à {df['Date'].max()}")
df.head(3)

Dimensions : 12907 lignes, 26 colonnes
Période couverte : 2018-01 à 2026-09


,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Commentaire annulations,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,Commentaire retards au départ,Nombre de trains en retard à l'arrivée,Retard moyen des trains en retard à l'arrivée,Retard moyen de tous les trains à l'arrivée,Commentaire retards à l'arrivée,Nombre trains en retard > 15min,Retard moyen trains en retard > 15 (si liaison concurrencée par vol),Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141,870,5,NaN,289,11.247809,3.693179,NaN,147,28.436735,6.511118,NaN,110,6.511118,44,8,36.134454,31.092437,10.924370,15.966387,5.042017,0.840336
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56,406,1,NaN,213,8.479969,4.567119,NaN,105,18.049048,5.363539,"Ce mois-ci, l'OD a été touchée par les incidents suivants :\nLe 1er : Tempête Carmen sur la façade Atlantique (52 TGV ; 1079mn)\nLe 3 : Tempête Eleonor sur l’ouest de la France (35 TGV ; 407mn)\nLe 5 : Dérangement d’une aiguille en gare de Massy TGV (57 TGV ; 1254mn)\nLe 7 : Dérangement du poste d’aiguillage de Paris Montparnasse (70 TGV ; 1490mn)\nLe 12 : Colis suspect en gare de Paris Montparnasse (32 TGV ; 499mn)\nLe 12 : Incident caténaire en gare de Rennes (26 TGV ; 2188mn)\nLe 17 : Dérangement du poste d’aiguillage de Paris Montparnasse (26 TGV ; 386mn)\nLe 17 : Présence d’une personne dans les voies à Ste Luce (11 TGV ; 212mn)\nLe 26 : Heurt d’une personne au niveau de Tiercé (44 TGV ; 1578mn)",32,5.363539,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166,226,0,NaN,21,6.239683,0.286283,NaN,19,24.736842,2.938053,NaN,11,2.938053,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111


## 1. État des lieux et qualité des données

On commence par vérifier la présence éventuelle de doublons, de valeurs manquantes et de valeurs négatives dans les colonnes numériques.


In [ ]:
doublons = df.duplicated().sum()
print(f"Nombre de lignes dupliquées : {doublons}")

na_counts = df.isna().sum()
df_na = pd.DataFrame(
    {
        "Valeurs manquantes": na_counts[na_counts > 0],
        "Part (%)": (na_counts[na_counts > 0] / len(df) * 100).round(2),
    }
)
display(df_na)

cols_num = df.select_dtypes(include=[np.number]).columns
neg_counts = (df[cols_num] < 0).sum()
df_neg = pd.DataFrame({"Valeurs négatives": neg_counts[neg_counts > 0]})
display(df_neg)

Nombre de lignes dupliquées : 0


,Valeurs manquantes,Part (%)
Commentaire annulations,12907,100.00
Commentaire retards au départ,12907,100.00
Commentaire retards à l'arrivée,12209,94.59


,Valeurs négatives
Retard moyen de tous les trains au départ,170
Retard moyen des trains en retard à l'arrivée,2
Retard moyen de tous les trains à l'arrivée,122
Retard moyen trains en retard > 15 (si liaison concurrencée par vol),8
Nombre trains en retard > 30min,43


On observe trois points principaux :

- **Doublons :** aucune ligne dupliquée dans le fichier.
- **Valeurs manquantes :** elles se concentrent sur les trois colonnes de commentaires textuels, qui sont vides à plus de 94 % et donc inexploitables pour l'analyse quantitative.
- **Valeurs négatives :** cinq colonnes sont concernées, mais elles ne relèvent pas toutes d'erreurs. Un retard moyen négatif calculé sur *tous* les trains (au départ ou à l'arrivée) traduit simplement des trains partis ou arrivés en avance. En revanche, un effectif négatif (`Nombre trains en retard > 30min`) ou un retard moyen négatif parmi des trains *en retard* sont de vraies anomalies à passer en `NaN`.


In [ ]:
df_clean = df.copy()

colonnes_a_supprimer = [
    "Commentaire annulations",
    "Commentaire retards au départ",
    "Commentaire retards à l'arrivée",
]
df_clean = df_clean.drop(columns=colonnes_a_supprimer)

cols_neg_aberrantes = [
    "Nombre trains en retard > 30min",
    "Retard moyen trains en retard > 15 (si liaison concurrencée par vol)",
    "Retard moyen des trains en retard à l'arrivée",
]
for col in cols_neg_aberrantes:
    df_clean.loc[df_clean[col] < 0, col] = np.nan

df_clean["Liaison"] = df_clean["Gare de départ"] + " -> " + df_clean["Gare d'arrivée"]

print(
    f"Table nettoyée : {df_clean.shape[0]} lignes, {df_clean.shape[1]} colonnes, "
    f"{df_clean['Liaison'].nunique()} liaisons uniques."
)

Table nettoyée : 12907 lignes, 24 colonnes, 130 liaisons uniques.


## 2. Vérification des incohérences métier sur les circulations

Avant de calculer des taux d'annulation et de retard, on vérifie les incohérences logiques sur les volumes : circulations prévues nulles ou négatives, annulations supérieures aux trains prévus, ou nombre de trains en retard supérieur aux trains ayant réellement circulé.


In [ ]:
trains_circules = (
    df_clean["Nombre de circulations prévues"] - df_clean["Nombre de trains annulés"]
)

prevus_invalides = df_clean["Nombre de circulations prévues"] <= 0
annulations_negatives = df_clean["Nombre de trains annulés"] < 0
annulations_sup_prevus = (
    df_clean["Nombre de trains annulés"] > df_clean["Nombre de circulations prévues"]
)
retards_sup_circules = (trains_circules >= 0) & (
    df_clean["Nombre de trains en retard à l'arrivée"] > trains_circules
)

bilan_incoherences = pd.Series(
    {
        "Circulations prévues <= 0": int(prevus_invalides.sum()),
        "Annulations < 0": int(annulations_negatives.sum()),
        "Annulations > trains prévus": int(annulations_sup_prevus.sum()),
        "Retards > trains circulés": int(retards_sup_circules.sum()),
    }
)
display(bilan_incoherences)

masque_valide = (
    ~prevus_invalides
    & ~annulations_negatives
    & ~annulations_sup_prevus
    & ~retards_sup_circules
)
df_valide = df_clean.loc[masque_valide].copy()
df_valide["trains_circules"] = trains_circules.loc[masque_valide]
print(f"Observations exploitables retenues : {len(df_valide)} / {len(df_clean)}")

Observations exploitables retenues : 12834 / 12907


,0
Circulations prévues <= 0,73
Annulations < 0,0
Annulations > trains prévus,63
Retards > trains circulés,0


## 3. Statistiques descriptives et choix des taux pondérés

On calcule les taux mensuels d'annulation et de retard à l'arrivée, puis on compare la moyenne simple des taux mensuels avec le taux pondéré par le nombre de trains. Une liaison qui ne fait circuler que 10 trains dans le mois ne doit pas peser autant qu'un grand axe à 500 trains dans les calculs agrégés.


In [ ]:
df_valide["taux_annulation"] = (
    df_valide["Nombre de trains annulés"]
    / df_valide["Nombre de circulations prévues"]
    * 100
)
df_valide["taux_retard_arrivee"] = np.where(
    df_valide["trains_circules"] > 0,
    df_valide["Nombre de trains en retard à l'arrivée"]
    / df_valide["trains_circules"]
    * 100,
    np.nan,
)

cols_resume = [
    "Nombre de circulations prévues",
    "Nombre de trains annulés",
    "Durée moyenne du trajet",
    "taux_annulation",
    "taux_retard_arrivee",
]
display(df_valide[cols_resume].describe().round(2))

taux_annul_simple = df_valide["taux_annulation"].mean()
taux_annul_pondere = (
    df_valide["Nombre de trains annulés"].sum()
    / df_valide["Nombre de circulations prévues"].sum()
    * 100
)
taux_retard_simple = df_valide["taux_retard_arrivee"].mean()
taux_retard_pondere = (
    df_valide["Nombre de trains en retard à l'arrivée"].sum()
    / df_valide["trains_circules"].sum()
    * 100
)

print(
    f"Taux d'annulation -> moyenne simple : {taux_annul_simple:.2f} % | "
    f"taux pondéré : {taux_annul_pondere:.2f} %"
)
print(
    f"Taux de retard    -> moyenne simple : {taux_retard_simple:.2f} % | "
    f"taux pondéré : {taux_retard_pondere:.2f} %"
)

Taux d'annulation -> moyenne simple : 3.69 % | taux pondéré : 3.01 %
Taux de retard    -> moyenne simple : 14.93 % | taux pondéré : 14.66 %


,Nombre de circulations prévues,Nombre de trains annulés,Durée moyenne du trajet,taux_annulation,taux_retard_arrivee
count,12834.00,12834.00,12834.00,12834.00,12834.00
mean,275.00,8.28,171.66,3.69,14.93
std,183.48,21.86,87.02,8.36,8.44
min,1.00,0.00,35.00,0.00,0.00
25%,153.00,0.00,100.00,0.00,9.38
50%,232.00,2.00,164.00,0.69,13.36
75%,367.00,6.75,224.00,2.94,18.70
max,1104.00,297.00,786.00,92.68,100.00


## 4. Exploration graphique de la problématique

### 4.1 Distribution des volumes par liaison

On agrège les données par liaison (avec taux pondérés) pour observer l'hétérogénéité de l'offre entre les petites et les grandes liaisons.


In [ ]:
df_liaisons = df_valide.groupby("Liaison", as_index=False).agg(
    nb_mois=("Date", "count"),
    trains_prevus=("Nombre de circulations prévues", "sum"),
    trains_annules=("Nombre de trains annulés", "sum"),
    trains_circules=("trains_circules", "sum"),
    trains_retard=("Nombre de trains en retard à l'arrivée", "sum"),
    duree_moyenne=("Durée moyenne du trajet", "mean"),
)
df_liaisons["taux_annulation"] = (
    df_liaisons["trains_annules"] / df_liaisons["trains_prevus"] * 100
)
df_liaisons["taux_retard_arrivee"] = np.where(
    df_liaisons["trains_circules"] > 0,
    df_liaisons["trains_retard"] / df_liaisons["trains_circules"] * 100,
    np.nan,
)

display(df_liaisons.sort_values("trains_prevus", ascending=False).head(10).round(2))

fig_hist = px.histogram(
    df_liaisons,
    x="trains_prevus",
    nbins=30,
    labels={"trains_prevus": "Trains prévus cumulés sur la période"},
    title="Distribution du volume cumulé de circulations par liaison",
)
fig_hist.show()

,Liaison,nb_mois,trains_prevus,trains_annules,trains_circules,trains_retard,duree_moyenne,taux_annulation,taux_retard_arrivee
9,BORDEAUX ST JEAN -> PARIS MONTPARNASSE,105,94952,3190,91762,14775,143.25,3.36,16.10
88,PARIS MONTPARNASSE -> BORDEAUX ST JEAN,105,91415,2830,88585,9753,144.30,3.10,11.01
114,RENNES -> PARIS MONTPARNASSE,105,74143,2224,71919,11648,100.20,3.00,16.20
96,PARIS MONTPARNASSE -> RENNES,105,73532,2080,71452,7897,99.43,2.83,11.05
33,LYON PART DIEU -> PARIS LYON,105,65817,1492,64325,7354,120.13,2.27,11.43
74,PARIS LYON -> LYON PART DIEU,105,64125,1460,62665,5242,120.16,2.28,8.37
28,LILLE -> PARIS NORD,105,60962,2035,58927,9612,68.86,3.34,16.31
105,PARIS NORD -> LILLE,105,60301,2008,58293,7824,67.23,3.33,13.42
50,NANTES -> PARIS MONTPARNASSE,105,58227,1712,56515,7305,133.30,2.94,12.93
93,PARIS MONTPARNASSE -> NANTES,105,55803,1625,54178,6305,131.39,2.91,11.64


<Plotly Figure: Distribution du volume cumulé de circulations par liaison>

### 4.2 Volume de desserte vs taux d'annulation

On croise le volume cumulé de trains prévus avec le taux d'annulation pondéré et on mesure la corrélation de Pearson et de Spearman (moins sensible aux valeurs extrêmes).


In [ ]:
pearson = df_liaisons["trains_prevus"].corr(
    df_liaisons["taux_annulation"], method="pearson"
)
spearman = (
    df_liaisons["trains_prevus"]
    .rank()
    .corr(df_liaisons["taux_annulation"].rank(), method="pearson")
)
print(
    f"Corrélation volume vs taux d'annulation -> Pearson : {pearson:.3f} | Spearman : {spearman:.3f}"
)

fig_scatter = px.scatter(
    df_liaisons,
    x="trains_prevus",
    y="taux_annulation",
    size="nb_mois",
    color="duree_moyenne",
    hover_name="Liaison",
    labels={
        "trains_prevus": "Trains prévus cumulés",
        "taux_annulation": "Taux d'annulation pondéré (%)",
        "duree_moyenne": "Durée moy. (min)",
    },
    title="Volume prévu vs taux d'annulation pondéré par liaison",
)
fig_scatter.show()

Corrélation volume vs taux d'annulation -> Pearson : -0.082 | Spearman : -0.125


<Plotly Figure: Volume prévu vs taux d'annulation pondéré par liaison>

### 4.3 Dispersion selon la catégorie de volume et lien avec les retards

On répartit les liaisons en trois groupes de volume équilibrés (tertiles) afin de comparer la dispersion des taux d'annulation, puis on croise taux d'annulation et taux de retard à l'arrivée.


In [ ]:
df_liaisons["groupe_volume"] = pd.qcut(
    df_liaisons["trains_prevus"],
    q=3,
    labels=["Faible volume", "Volume moyen", "Fort volume"],
)

fig_box = px.box(
    df_liaisons,
    x="groupe_volume",
    y="taux_annulation",
    color="groupe_volume",
    points="all",
    hover_name="Liaison",
    labels={
        "groupe_volume": "Catégorie de desserte (tertiles)",
        "taux_annulation": "Taux d'annulation pondéré (%)",
    },
    title="Dispersion du taux d'annulation selon le volume de la liaison",
)
fig_box.show()

fig_croise = px.scatter(
    df_liaisons,
    x="taux_annulation",
    y="taux_retard_arrivee",
    size="trains_prevus",
    color="duree_moyenne",
    hover_name="Liaison",
    labels={
        "taux_annulation": "Taux d'annulation pondéré (%)",
        "taux_retard_arrivee": "Taux de retard à l'arrivée pondéré (%)",
        "duree_moyenne": "Durée moy. (min)",
    },
    title="Croisement taux d'annulation vs taux de retard à l'arrivée",
)
fig_croise.show()

<Plotly Figure: Dispersion du taux d'annulation selon le volume de la liaison>

<Plotly Figure: Croisement taux d'annulation vs taux de retard à l'arrivée>

## 5. Synthèse

- Le jeu de données ne comporte pas de doublons, mais demande d'écarter les colonnes de commentaires quasi vides, de distinguer les retards négatifs normaux (avances) des anomalies, et de filtrer les incohérences de volumes.
- L'utilisation de taux pondérés par le nombre de trains prévus ou circulés évite que des liaisons très peu desservies ne faussent les moyennes.
- Les liaisons très fréquentées concentrent logiquement davantage d'annulations en nombre absolu, mais ne présentent pas de hausse systématique du taux relatif ; la variabilité est plus importante sur les liaisons à faible volume.
- Ces traitements et indicateurs sont repris et testés dans `src/projet_railinsight/circulations.py`, puis intégrés dans la page Streamlit `pages/02_Circulations.py`.
